# コニカルホーン解析計算（HFSS 不使用）

円形導波管 TE11 給電コニカルホーンを、球面固有モードと開口面スペクトルの数値積分から評価します。単位は SI です。負荷 Model A は自由空間インピーダンス、Model B は伝搬波・エバネセント波を含む複素放射アドミタンスです。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import constants

from conical_horn import ConicalHorn, HornGeometry

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True})

## 形状と固有値

In [ ]:
geometry = HornGeometry(
    waveguide_radius=12e-3,
    aperture_radius=35e-3,
    flare_half_angle=np.deg2rad(15),
)
horn = ConicalHorn(geometry, radial_order=220, power_order=180)
print(f"TE11 cutoff = {geometry.cutoff_frequency/1e9:.4f} GHz")
print(f"R1 = {geometry.throat_radius*1e3:.3f} mm")
print(f"R2 = {geometry.slant_length*1e3:.3f} mm")
print(f"axial length = {geometry.axial_length*1e3:.3f} mm")
print(f"spherical eigenvalue nu1 = {horn.nu:.9f}")

## インピーダンスと S11

Model B はスペクトル積分を各周波数で行うため、Model A より時間がかかります。まず 41 点で全体を確認し、必要なら点数を増やしてください。

In [ ]:
frequencies = np.linspace(8e9, 13e9, 41)
sweep_a = horn.frequency_sweep(frequencies, load_model="A")
sweep_b = horn.frequency_sweep(frequencies, load_model="B")

In [ ]:
fig, ax = plt.subplots()
ax.plot(frequencies/1e9, sweep_a["S11_dB"], label="Model A: $Z_L=\eta_0$")
ax.plot(frequencies/1e9, sweep_b["S11_dB"], label="Model B: spectral admittance")
ax.set(xlabel="Frequency (GHz)", ylabel="$|S_{11}|$ (dB)", ylim=(-50, 0))
ax.legend(); plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(8, 7))
fghz = frequencies/1e9
ax1.plot(fghz, sweep_b["Zin"].real, label="Re $Z_{in}$")
ax1.plot(fghz, sweep_b["Zg"].real, "--", label="$Z_g$")
ax2.plot(fghz, sweep_b["Zin"].imag, label="Im $Z_{in}$")
ax1.set(ylabel="Resistance ($\Omega$)"); ax2.set(xlabel="Frequency (GHz)", ylabel="Reactance ($\Omega$)")
ax1.legend(); ax2.legend(); plt.show()

## ビーム、HPBW、交差偏波

In [ ]:
f0 = 10e9
beam = horn.beam(f0)
theta_deg = np.rad2deg(beam["theta"])
fig, ax = plt.subplots()
ax.plot(theta_deg, beam["E_dB"], label="E plane")
ax.plot(theta_deg, beam["H_dB"], label="H plane")
ax.plot(theta_deg, beam["cross_dB"], label="Cross-pol, $\phi=45^\circ$")
ax.set(xlabel="$\theta$ (degree)", ylabel="Normalized field (dB)", xlim=(-90, 90), ylim=(-60, 0))
ax.legend(); plt.show()
print(f"E-plane HPBW = {np.rad2deg(horn.hpbw(beam['theta'], beam['E_dB'])):.2f} deg")
print(f"H-plane HPBW = {np.rad2deg(horn.hpbw(beam['theta'], beam['H_dB'])):.2f} deg")

## 軸上指向性と実現利得

In [ ]:
fig, ax = plt.subplots()
ax.plot(fghz, 10*np.log10(sweep_b["directivity"]), label="Directivity")
ax.plot(fghz, 10*np.log10(np.maximum(sweep_b["realized_gain"], 1e-15)), label="Realized gain")
ax.set(xlabel="Frequency (GHz)", ylabel="dBi")
ax.legend(); plt.show()

## 基本検証

位相誤差をゼロにした TE11 円形開口能率、および遠方での外向き球面波インピーダンスを確認します。

In [ ]:
from conical_horn import CHI_11_PRIME, ETA_0
efficiency = 2/(CHI_11_PRIME**2 - 1)
x = 1000.0
h = horn.schelkunoff_hankel(horn.nu, x, 2)
hp = horn.schelkunoff_hankel_derivative(horn.nu, x, 2)
print(f"zero-phase aperture efficiency = {efficiency:.6f} (target 0.8368)")
print(f"outgoing-wave impedance at kr=1000 = {-1j*ETA_0*h/hp:.4f} ohm")